In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import torch

print(f"PyTorch: {torch.__version__}")
print(f"GPU var mı: {torch.cuda.is_available()}")

PyTorch: 2.11.0+cpu
GPU var mı: False


In [2]:
# Veri yükleme
kaggle_df = pd.read_csv("../data/raw/train.csv")[["text", "label"]]
haber_df = pd.read_csv("../data/raw/haberler_etiketli.csv")[["baslik", "etiket"]].rename(columns={"baslik": "text", "etiket": "label"})
synthetic_df = pd.read_csv("../data/raw/synthetic_data.csv")[["cumle", "kategori"]].rename(columns={"cumle": "text", "kategori": "label"})

# Etiketleri standardize et
etiket_map = {"Positive": "bullish", "Negative": "bearish", "Notr": "neutral"}
kaggle_df["label"] = kaggle_df["label"].map(etiket_map)

# Birleştir
df = pd.concat([kaggle_df, haber_df, synthetic_df], ignore_index=True)
df = df.dropna()

print(f"Kaggle: {len(kaggle_df)}")
print(f"Haberler: {len(haber_df)}")
print(f"Synthetic: {len(synthetic_df)}")
print(f"Toplam: {len(df)}")
print(df["label"].value_counts())

Kaggle: 440679
Haberler: 299
Synthetic: 100000
Toplam: 540978
label
bullish    269340
neutral    187350
bearish     84288
Name: count, dtype: int64


In [3]:
# Etiketleri sayıya çevir
etiket_map_sayı = {"bearish": 0, "bullish": 1, "neutral": 2}
df["label_id"] = df["label"].map(etiket_map_sayı)

# Finansal veriyi ağırlıklandır
df["weight"] = df["label"].apply(
    lambda x: 1  # hepsi eşit ağırlık — artık yeterli finansal veri var
)

# Train/test split
train_df, test_df = train_test_split(
    df, test_size=0.1, random_state=42, stratify=df["label"]
)

print(f"Train: {len(train_df)}")
print(f"Test: {len(test_df)}")
print("\nTrain dağılımı:")
print(train_df["label"].value_counts())

Train: 486880
Test: 54098

Train dağılımı:
label
bullish    242406
neutral    168615
bearish     75859
Name: count, dtype: int64


In [4]:
# Veri yükleme
kaggle_df = pd.read_csv("../data/raw/train.csv")[["text", "label"]]
haber_df = pd.read_csv("../data/raw/haberler_etiketli.csv")[["baslik", "etiket"]].rename(columns={"baslik": "text", "etiket": "label"})
synthetic_df = pd.read_csv("../data/raw/synthetic_data.csv")[["cumle", "kategori"]].rename(columns={"cumle": "text", "kategori": "label"})

# Etiketleri standardize et
etiket_map = {"Positive": "bullish", "Negative": "bearish", "Notr": "neutral"}
kaggle_df["label"] = kaggle_df["label"].map(etiket_map)

# Kaynak ekle
kaggle_df["kaynak"] = "kaggle"
haber_df["kaynak"] = "gercek"
synthetic_df["kaynak"] = "synthetic"

# Birleştir
df = pd.concat([kaggle_df, haber_df, synthetic_df], ignore_index=True)
df = df.dropna()

# Ağırlık ekle
def agirlik(kaynak):
    if kaynak == "gercek":
        return 10    # gerçek haberler en değerli
    elif kaynak == "synthetic":
        return 2     # finansal ama yapay
    else:
        return 1     # genel Türkçe

df["weight"] = df["kaynak"].apply(agirlik)

# Ağırlıklı örnekleme — gerçek ve synthetic haberleri çoğalt
gercek = pd.concat([haber_df] * 10, ignore_index=True)
synthetic_weighted = pd.concat([synthetic_df] * 2, ignore_index=True)

df_final = pd.concat([kaggle_df, gercek, synthetic_weighted], ignore_index=True)
df_final = df_final.dropna()

print(f"Kaggle: {len(kaggle_df)}")
print(f"Gerçek haberler (10x): {len(gercek)}")
print(f"Synthetic (2x): {len(synthetic_weighted)}")
print(f"Toplam: {len(df_final)}")
print(df_final["label"].value_counts())

Kaggle: 440679
Gerçek haberler (10x): 2990
Synthetic (2x): 200000
Toplam: 643669
label
bullish    303187
neutral    222411
bearish    118071
Name: count, dtype: int64


In [5]:
# Etiketleri sayıya çevir
etiket_map_sayi = {"bearish": 0, "bullish": 1, "neutral": 2}
df_final["label_id"] = df_final["label"].map(etiket_map_sayi)

# Train/test split
train_df, test_df = train_test_split(
    df_final[["text", "label_id"]],
    test_size=0.1,
    random_state=42,
    stratify=df_final["label_id"]
)

print(f"Train: {len(train_df)}")
print(f"Test: {len(test_df)}")

Train: 579302
Test: 64367


In [6]:
# Tokenizer yükle
model_adi = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_adi)

# Dataset'e çevir
train_dataset = Dataset.from_pandas(
    train_df.rename(columns={"label_id": "labels"})
)
test_dataset = Dataset.from_pandas(
    test_df.rename(columns={"label_id": "labels"})
)

# Tokenize et
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

print("Tokenize tamamlandı.")
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

Map:   0%|          | 0/579302 [00:00<?, ? examples/s]

Map:   0%|          | 0/64367 [00:00<?, ? examples/s]

Tokenize tamamlandı.
Train: 579302, Test: 64367


In [7]:
# Model yükle
model = AutoModelForSequenceClassification.from_pretrained(
    model_adi,
    num_labels=3
)

# Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=200,
    warmup_steps=500,
    weight_decay=0.01,
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    report = classification_report(
        labels, preds,
        target_names=["bearish", "bullish", "neutral"]
    )
    print(report)
    return {"f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Eğitime hazır.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Eğitime hazır.


In [ ]:
trainer.train()

C:\Users\Admin\PycharmProjects\turkish-financial-sentiment\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
